In [13]:
#Moderate
#Focus: Email validation, normalization, dedupe, primary flag.
#Uses (only): Person.EmailAddress
#Creates: silver_person_email and silver_adw.person_email_invalid

In [14]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable


silver_base_path = "abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_person_email/"
silver_base_path_invalid = "abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Files/silver/silver_person_email_invalid/"
source_system = "AdventureWorks"

base = "abfss://Fabmigration@onelake.dfs.fabric.microsoft.com/Synapsemigration.Lakehouse/Tables/fabmigation1/bronze/adventureworks/Person"

tables = ["EmailAddress"]

dfs = {t: spark.read.parquet(f"{base}.{t}.parquet/") for t in tables}

dfEmailAddress = dfs["EmailAddress"]
display(dfs["EmailAddress"])

In [15]:

def parquet_exists(path: str) -> bool:
    try:
        from notebookutils import mssparkutils
        return mssparkutils.fs.exists(path)
    except Exception:
        try:
            spark.read.parquet(path).limit(1).count()
            return True
        except Exception:
            return False

def read_parquet_if_exists(path: str):
    if parquet_exists(path):
        return spark.read.parquet(path)
    return None


In [16]:
# Normalize, validate, dedup, primary
email = (
    dfEmailAddress
    .withColumn("email_trim", F.trim(F.col("EmailAddress")))
    .withColumn("email_lower", F.lower(F.col("email_trim")))
    .withColumn("domain", F.lower(F.regexp_extract(F.col("email_lower"), r"@(.+)$", 1)))
)
display(email.limit(50))

In [17]:
# Simple validity (approx RFC): local@domain with basic allowed chars and a TLD
regex_valid = r"^[A-Za-z0-9.!#$%&'*+/=?^_`{|}~-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+$"
email = email.withColumn("is_valid", F.col("email_lower").rlike(regex_valid))
display(email.limit(50))

In [18]:
# Domain category (free vs corporate) - small reference
free_domains = F.array([F.lit(d) for d in ["gmail.com","yahoo.com","hotmail.com","outlook.com","live.com","icloud.com","aol.com","msn.com"]])
email = email.withColumn("domain_category",
                         F.when(F.array_contains(free_domains, F.col("domain")), F.lit("free"))
                          .when(F.col("domain").isNull(), F.lit(None))
                          .otherwise(F.lit("corporate")))
display(email.limit(50))

In [19]:
# Split valid/invalid for observability
email_valid = email.filter(F.col("is_valid") == True)
email_invalid = email.filter(F.col("is_valid") == False)

In [20]:
# Dedupe by (BusinessEntityID, email_lower) keeping latest ModifiedDate
w = Window.partitionBy("BusinessEntityID", "email_lower").orderBy(F.col("ModifiedDate").desc_nulls_last())
email_dedup = (
    email_valid
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)
display(email_dedup.limit(50))

In [21]:
# Identify a primary email per BusinessEntityID (latest ModifiedDate)
w2 = Window.partitionBy("BusinessEntityID").orderBy(F.col("ModifiedDate").desc_nulls_last(), F.col("EmailAddressID").asc())
email_final = (
    email_dedup
    .withColumn("is_primary", (F.row_number().over(w2) == 1).cast("boolean"))
    .withColumn("person_email_sk", F.sha2(F.concat_ws("||",
                                                      F.col("BusinessEntityID").cast("string"),
                                                      F.col("email_lower")), 256))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("record_source", F.lit("Person.EmailAddress"))
    .withColumn("source_system", F.lit(source_system))
    .select(
        "person_email_sk", "BusinessEntityID", F.col("EmailAddressID").alias("email_id"),
        F.col("email_lower").alias("email"), "domain", "domain_category",
        "is_primary", "ModifiedDate", "record_source", "source_system", "ingestion_timestamp"
    )
)
display(email_final.limit(10))

In [22]:
invalid_out = (
    email_invalid
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .select(
        "BusinessEntityID",
        F.col("EmailAddressID").alias("email_id"),
        F.col("email_trim").alias("raw_email"),
        "ModifiedDate",
        "ingestion_timestamp"
    )
)
display(invalid_out.limit(50))

In [23]:
existing_valid = read_parquet_if_exists(silver_base_path)
print("existing_valid:", existing_valid)
if existing_valid is None:
    valid_output = email_final
else:
    merged = (
        existing_valid.unionByName(email_final, allowMissingColumns=True)
        .withColumn("nk", F.concat_ws("||", F.col("BusinessEntityID").cast("string"), F.lower(F.col("email"))))
    )
    wv = Window.partitionBy("nk").orderBy(F.col("ModifiedDate").desc_nulls_last())
    valid_output = merged.withColumn("rn", F.row_number().over(wv)).filter("rn = 1").drop("rn", "nk")
display(valid_output.limit(50))

In [24]:
valid_output.write.mode("overwrite").parquet(silver_base_path)
invalid_out.write.mode("overwrite").parquet(silver_base_path_invalid)